In [0]:
SHOW TABLES;
DESCRIBE DETAIL hive_metastore.default.sales_demo;
DESCRIBE HISTORY hive_metastore.default.sales_demo;
USE hive_metastore.silver_demo.accounts_clean;
-- Add 100 rows with unique AccountID continuing after the current max
WITH base AS (
  SELECT COALESCE(MAX(AccountID), 0) AS max_id FROM hive_metastore.silver_demo.accounts_clean
)
INSERT INTO hive_metastore.silver_demo.accounts_clean (AccountID, AccountName, Country)
SELECT
  b.max_id + r.id + 1                                            AS AccountID,         -- 1..100 after current max
  CONCAT('Account_', CAST(b.max_id + r.id + 1 AS STRING))         AS AccountName,
  element_at(
    array('USA','Canada','UK','Germany','France'),
    CAST(FLOOR(RAND() * 5) AS INT) + 1                            -- random 1..5
  )                                                               AS Country
FROM base b
CROSS JOIN range(100) AS r;     -- r.id = 0..99


In [0]:
-- Insert 100 random rows into accounts_clean
WITH base AS (
  SELECT COALESCE(MAX(AccountID), 0) AS max_id FROM hive_metastore.silver_demo.accounts_clean
)
INSERT INTO hive_metastore.silver_demo.accounts_clean (AccountID, AccountName, Country)
SELECT
  base.max_id + r.id + 1 AS AccountID,
  CONCAT('Account_', CAST(base.max_id + r.id + 1 AS STRING)) AS AccountName,
  element_at(
    array('USA', 'Canada', 'UK', 'Germany', 'France'),
    CAST(FLOOR(rand() * 5) AS INT) + 1
  ) AS Country
FROM base
CROSS JOIN range(100) AS r;



In [0]:
SELECT * FROM hive_metastore.silver_demo.accounts_clean


In [0]:
-- Using a loop construct in Databricks SQL (via scripting)
BEGIN
  FOR i IN RANGE(103) DO
    UPDATE silver_demo.accounts_clean
    SET AccountID = AccountID + 1
    WHERE AccountID = 'A001';
  END FOR;
END;


In [0]:
UPDATE silver_demo.accounts_clean
SET AccountID = AccountID + 103
WHERE AccountID = 'A001';

In [0]:
UPDATE silver_demo.accounts_clean
SET AccountID = CAST(FLOOR(RAND() * 76) + 1 AS INT)
WHERE Country IN ('USA', 'Canada', 'France', 'Germany', 'UNKNOWN');



In [0]:
CREATE OR REPLACE TABLE silver_demo.accounts_clean AS
SELECT
  CASE
    WHEN Country IN ('USA', 'Canada', 'France', 'Germany', 'UNKNOWN')
      THEN CAST(FLOOR(RAND() * 76) + 1 AS INT)
    ELSE AccountID
  END AS AccountID,
  AccountName,
  Country
FROM silver_demo.accounts_clean;

SELECT * FROM silver_demo.accounts_clean


In [0]:
UPDATE silver_demo.accounts_clean
SET AccountName = element_at(
    array('Walmart', 'Target', 'Loblaws', 'SportChek', 'Messer',
          'Lidl', 'Mosmart', 'Frazier', 'Shoppers', 'Hortons'),
    CAST(FLOOR(RAND() * 10) AS INT) + 1
);


In [0]:
%python
from pyspark.sql.functions import floor, rand, element_at, array

# Define the array of names
names = array(
    'Walmart', 'Target', 'Loblaws', 'SportChek', 'Messer',
    'Lidl', 'Mosmart', 'Frazier', 'Shoppers', 'Hortons'
)

# Load the table
df = spark.table("silver_demo.accounts_clean")

# Assign a random AccountName from the list
df_updated = df.withColumn(
    "AccountName",
    element_at(
        names,
        (floor(rand() * 10) + 1).cast("int")
    )
)

# Overwrite the table with updated values
df_updated.write.format("delta").mode("overwrite").saveAsTable("silver_demo.accounts_clean")

display(df_updated)

In [0]:
-- Use a CTE to assign sequence numbers, then update with cyclic names
WITH numbered AS (
  SELECT 
    AccountID,  -- replace with your actual unique key column
    ROW_NUMBER() OVER (ORDER BY AccountID) AS seq_num
  FROM silver_demo.accounts_clean
)

UPDATE silver_demo.accounts_clean AS a
SET a.AccountName = CASE ((n.seq_num - 1) % 10) + 1
    WHEN 1 THEN 'Walmart'
    WHEN 2 THEN 'Target'
    WHEN 3 THEN 'Loblaws'
    WHEN 4 THEN 'SportChek'
    WHEN 5 THEN 'Messer'
    WHEN 6 THEN 'Lidl'
    WHEN 7 THEN 'Mosmart'
    WHEN 8 THEN 'Frazier'
    WHEN 9 THEN 'Shoppers'
    WHEN 10 THEN 'Hortons'
  END
FROM numbered n
WHERE a.AccountID = n.AccountID;


In [0]:
MERGE INTO silver_demo.accounts_clean AS target
USING (
  SELECT
    AccountID,
    CASE ((ROW_NUMBER() OVER (ORDER BY AccountID) - 1) % 10) + 1
      WHEN 1 THEN 'Walmart'
      WHEN 2 THEN 'Target'
      WHEN 3 THEN 'Loblaws'
      WHEN 4 THEN 'SportChek'
      WHEN 5 THEN 'Messer'
      WHEN 6 THEN 'Lidl'
      WHEN 7 THEN 'Mosmart'
      WHEN 8 THEN 'Frazier'
      WHEN 9 THEN 'Shoppers'
      WHEN 10 THEN 'Hortons'
    END AS new_name
  FROM silver_demo.accounts_clean
) AS src
ON target.AccountID = src.AccountID
WHEN MATCHED THEN UPDATE SET target.AccountName = src.new_name;

In [0]:
UPDATE silver_demo.accounts_clean
SET AccountName = CASE ((ROW_NUMBER() OVER (ORDER BY AccountID) - 1) % 10) + 1
    WHEN 1 THEN 'Walmart'
    WHEN 2 THEN 'Target'
    WHEN 3 THEN 'Loblaws'
    WHEN 4 THEN 'SportChek'
    WHEN 5 THEN 'Messer'
    WHEN 6 THEN 'Lidl'
    WHEN 7 THEN 'Mosmart'
    WHEN 8 THEN 'Frazier'
    WHEN 9 THEN 'Shoppers'
    WHEN 10 THEN 'Hortons'
END

In [0]:
UPDATE silver_demo.accounts_clean 
SET AccountName = 'Walmart'
WHERE AccountID BETWEEN 1 AND 10


-- 'Target'
-- 'Loblaws'
-- 'SportChek'
-- 'Messer'
-- 'Lidl'
-- 'Mosmart'
-- 'Frazier'
-- 'Shoppers'
-- 'Hortons'

In [0]:
UPDATE silver_demo.accounts_clean
SET AccountName = CASE
    WHEN AccountID BETWEEN 1 AND 10 THEN 'Walmart'
    WHEN AccountID BETWEEN 11 AND 20 THEN 'Loblaws'
    WHEN AccountID BETWEEN 21 AND 30 THEN 'SportChek'
    WHEN AccountID BETWEEN 31 AND 40 THEN 'Messer'
    WHEN AccountID BETWEEN 41 AND 50 THEN 'Lidl'
    WHEN AccountID BETWEEN 51 AND 60 THEN 'Mosmart'
    WHEN AccountID BETWEEN 61 AND 70 THEN 'Frazier'
    WHEN AccountID BETWEEN 71 AND 80 THEN 'Shoppers'
    WHEN AccountID BETWEEN 81 AND 90 THEN 'Hortons'
    ELSE 'Target'
  END